# PropCare AI — Stage 3: Deep Agent Property Operations Platform

Stage 2 offers precise control, but manually expanding supervisor routes, state fields, and specialist coordination becomes more complex as requests become broader. Stage 3 uses a Deep Agent coordinator that plans and delegates to focused subagents, while returning a safe structured result rather than hidden reasoning.

## Safe execution flow

**Tenant → PropCare Deep Agent → plan/delegate → Maintenance / Billing / Resident Services subagents → synthesis → structured resolution**. This notebook shows safe metadata, configured tools, and retrieved fictional records. It intentionally does not expose chain-of-thought or internal model reasoning.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'backend').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TENANT_ID = 'T-1001'
print(f'Project root: {ROOT}')

In [ ]:
from backend.stage3.propcare_deep_agent import AGENTS_PATH, Stage3Resolution, _subagents

print('AGENTS.md rules loaded from:', AGENTS_PATH)
print('\n'.join(AGENTS_PATH.read_text(encoding='utf-8').splitlines()))

subagent_metadata = [
    {
        'name': spec['name'],
        'description': spec['description'],
        'tools': [tool.name for tool in spec['tools']],
        'response_format': spec['response_format'].__name__,
    }
    for spec in _subagents()
]
subagent_metadata

## Existing tenant-safe services

The Deep Agent reuses the same service layer as the earlier stages. Runtime context supplies the authenticated tenant ID; a model is never trusted to invent or request it. These read-only calls show the kinds of evidence available to the specialist tools.

In [ ]:
from backend.services import property_service

tenant = property_service.lookup_tenant(TENANT_ID)
payment = property_service.check_rent_status(TENANT_ID)
open_requests = property_service.check_active_maintenance_requests(TENANT_ID)

print({
    'tenant': tenant.name if tenant else None,
    'unit_id': tenant.unit_id if tenant else None,
    'payment_status': payment.payment_status if payment else None,
    'active_request_ids': [request.request_id for request in open_requests],
})

## The actual Deep Agent factory

The production factory calls `create_deep_agent(...)` with `subagents=`, `Stage3Context`, `Stage3Resolution`, an `InMemorySaver` checkpointer, and the `AGENTS.md` memory file. The code below displays that small factory rather than duplicating the application implementation.

In [ ]:
import inspect
from backend.config import load_environment
from backend.stage3.propcare_deep_agent import build_stage3_agent

print(inspect.getsource(build_stage3_agent))
print('\nStructured response fields:', list(Stage3Resolution.model_fields))

load_environment()
if os.getenv('OPENAI_API_KEY'):
    stage3_agent = build_stage3_agent()
    print('Built Deep Agent type:', type(stage3_agent).__name__)
else:
    stage3_agent = None
    print('No OPENAI_API_KEY found: metadata and real read-only services remain runnable; add a key to build the live coordinator.')

## Suggested Stage 3 demonstrations

| Domain | Prompt | Expected safe result |
| --- | --- | --- |
| Billing | `Can you check whether my rent for this month has been paid?` | Billing specialist, payment status, no work order. |
| Maintenance | `The kitchen tap is leaking badly.` | Maintenance specialist creates or reuses an appropriate work order. |
| Compensation / multi-domain | `My heater has failed again and I want compensation.` | Maintenance and Billing findings, then a pending manager approval; no automatic credit. |
| Ambiguous / general | `Please investigate my account and property history.` | Resident Services gathers tenant, open-work-order, and billing context; synthesis reports real findings. |

In [ ]:
DEMO_PROMPTS = {
    'billing': 'Can you check whether my rent for this month has been paid?',
    'maintenance': 'The kitchen tap is leaking badly.',
    'compensation': 'My heater has failed again and I want compensation.',
    'ambiguous': 'Please investigate my account and property history.',
}
DEMO_PROMPTS

## Optional live coordinator run

`start_stage3()` can create or reuse a real local demo work order and may produce a pending approval. Leave the call opt-in in this portfolio notebook so it does not mutate shared data on an explanatory run. In the application, tenant requests reach it through the authenticated FastAPI boundary, which injects the tenant ID.

In [ ]:
from backend.stage3.propcare_deep_agent import start_stage3

RUN_LIVE_STAGE3 = False
if RUN_LIVE_STAGE3:
    thread_id, workflow = start_stage3(TENANT_ID, DEMO_PROMPTS['billing'])
    print({'thread_id': thread_id, 'status': workflow['status'], 'resolution': workflow['resolution']})
else:
    print('Live Stage 3 call skipped. Set RUN_LIVE_STAGE3 = True only when you intend to use local demo data and a configured provider key.')

## Stage 3 takeaway

Deep Agents supplies the coordinator/subagent abstraction and focused contexts, reducing manual graph wiring for broader operations. PropCare still keeps the critical product controls outside model discretion: authenticated tenant identity, tenant-safe service access, duplicate prevention, PKR financial context, and manager approval for any service credit.